In [1]:

import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Load dataset
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names

# 2. Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Normalize data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 4. Fitness function
def fitness(wolf):
    selected = wolf >= 0

    # Avoid selecting zero features
    if np.sum(selected) == 0:
        selected[np.argmax(wolf)] = True

    model = LogisticRegression(max_iter=2000)
    model.fit(X_train[:, selected], y_train)

    prediction = model.predict(X_test[:, selected])
    accuracy = accuracy_score(y_test, prediction)

    # Lower fitness is better
    return 1 - accuracy + 0.01 * np.mean(selected)


# 5. Initialize GWO parameters
wolves = 8
iterations = 20
dimensions = X.shape[1]

# Random wolf positions
positions = np.random.uniform(-1, 1, (wolves, dimensions))

# 6. Grey Wolf Optimization
for iteration in range(iterations):

    scores = np.array([fitness(w) for w in positions])

    # Identify Alpha, Beta and Delta
    order = np.argsort(scores)

    alpha = positions[order[0]].copy()
    beta = positions[order[1]].copy()
    delta = positions[order[2]].copy()

    # Decrease a from 2 to 0
    a = 2 - iteration * (2 / iterations)

    # Update wolf positions
    for i in range(wolves):

        A1 = 2 * a * np.random.rand(dimensions) - a
        C1 = 2 * np.random.rand(dimensions)
        D_alpha = abs(C1 * alpha - positions[i])
        X1 = alpha - A1 * D_alpha

        A2 = 2 * a * np.random.rand(dimensions) - a
        C2 = 2 * np.random.rand(dimensions)
        D_beta = abs(C2 * beta - positions[i])
        X2 = beta - A2 * D_beta

        A3 = 2 * a * np.random.rand(dimensions) - a
        C3 = 2 * np.random.rand(dimensions)
        D_delta = abs(C3 * delta - positions[i])
        X3 = delta - A3 * D_delta

        # Calculate new position
        positions[i] = (X1 + X2 + X3) / 3

        # Convert positions into binary values
        probability = 1 / (1 + np.exp(-np.clip(positions[i], -50, 50)))
        positions[i] = np.where(probability >= 0.5, 1, -1)

print("Optimization completed!")

# 7. Find the best feature combination
scores = np.array([fitness(w) for w in positions])
best = positions[np.argmin(scores)]

selected = best >= 0

# 8. Display selected features
print("\nSelected Features:")

for name in feature_names[selected]:
    print(name)

# 9. Train final model
final_model = LogisticRegression(max_iter=2000)
final_model.fit(X_train[:, selected], y_train)

prediction = final_model.predict(X_test[:, selected])
accuracy = accuracy_score(y_test, prediction)

print("\nTotal Features:", dimensions)
print("Selected Features:", np.sum(selected))
print("Final Accuracy:", round(accuracy * 100, 2), "%")


Optimization completed!

Selected Features:
mean radius
mean perimeter
mean compactness
perimeter error
area error
fractal dimension error
worst area
worst smoothness
worst concavity
worst fractal dimension

Total Features: 30
Selected Features: 10
Final Accuracy: 99.12 %
